# DSC2107 Group 8 — itch.io Top Sellers

## Notebook 1: Reconnaissance and a small test
Run this notebook before running the full scraper. It checks the site's current robots.txt, inspects the public HTML, and helps verify selectors. Do not continue to full collection if the page blocks requests or the selectors are not confirmed.

**Responsible collection:** public pages only; no login, CAPTCHA bypass, or private endpoints; minimum 3-second delay between page requests.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE_URL = 'https://itch.io/games/top-sellers'
ROBOTS_URL = 'https://itch.io/robots.txt'
USER_AGENT = 'DSC2107-Group8-StudentResearch (contact: cherissaiteriteka32@gmail.com)'

session = requests.Session()
session.headers.update({'User-Agent': USER_AGENT, 'Accept': 'text/html,application/xhtml+xml'})
print('Setup complete')
print('Target:', BASE_URL)
print('User-Agent:', USER_AGENT)

Setup complete
Target: https://itch.io/games/top-sellers
User-Agent: DSC2107-Group8-StudentResearch (contact: cherissaiteriteka32@gmail.com)


## 1. Check robots.txt before inspecting the page

In [2]:
robots_response = session.get(ROBOTS_URL, timeout=20)
print('robots.txt HTTP status:', robots_response.status_code)
robots_response.raise_for_status()
print(robots_response.text)

robots = RobotFileParser()
robots.set_url(ROBOTS_URL)
robots.parse(robots_response.text.splitlines())
allowed = robots.can_fetch(USER_AGENT, BASE_URL)
print('\nCan this User-Agent fetch the target page?', allowed)
if not allowed:
    raise PermissionError('robots.txt does not allow this target. Stop here.')

robots.txt HTTP status: 200
User-agent: *
Disallow: /embed/
Disallow: /embed-upload/
Disallow: /search
Disallow: /checkout/
Disallow: /game/download/
Disallow: /bundle/download/
Disallow: /register-for-purchase/
Disallow: /email-feedback/

Sitemap: https://itch.io/sitemap.xml


Can this User-Agent fetch the target page? True


## 2. Inspect one public HTML page (not the full scrape)

In [3]:
time.sleep(3)
response = session.get(BASE_URL, timeout=25)
print('HTTP status:', response.status_code)
print('Final URL:', response.url)
if response.status_code in (401, 403, 429):
    raise RuntimeError('Access denied or rate-limited. Stop and document the response.')
response.raise_for_status()
soup = BeautifulSoup(response.text, 'html.parser')
print('Page title:', soup.title.get_text(' ', strip=True) if soup.title else 'No title')
print('HTML characters:', len(response.text))
cells = soup.select('.game_cell')
print('Possible .game_cell containers:', len(cells))
for i, cell in enumerate(cells[:3], start=1):
    print(f'\\n--- Listing {i} ---')
    print('Classes:', cell.get('class'))
    print('Text:', cell.get_text(' ', strip=True)[:500])
    for link in cell.select('a[href]')[:8]:
        print('Link:', link.get_text(' ', strip=True), '| href:', link.get('href'), '| class:', link.get('class'))

HTTP status: 200
Final URL: https://itch.io/games/top-sellers
Page title: Top selling games - itch.io
HTML characters: 93984
Possible .game_cell containers: 36
\n--- Listing 1 ---
Classes: ['game_cell', 'has_cover', 'lazy_images']
Text: Add to collection The Choicer Voicer $5 Match your voice against your choice of audio clips in a mini-gameshow! YeahMaybe
Link:  | href: https://yeahmaybe.itch.io/the-choicer-voicer | class: ['thumb_link', 'game_link']
Link: Add to collection | href: /g/yeahmaybe/the-choicer-voicer/add-to-collection?source=browse | class: ['action_btn', 'add_to_collection_btn']
Link: The Choicer Voicer | href: https://yeahmaybe.itch.io/the-choicer-voicer | class: ['title', 'game_link']
Link: YeahMaybe | href: https://yeahmaybe.itch.io | class: None
\n--- Listing 2 ---
Classes: ['game_cell', 'has_cover', 'lazy_images']
Text: Add to collection The Annual Ghost Town Pumpkin Festival carve a lil pumpkin and share it with the world adamgryu Simulation
Link:  | href: https://

## 3. Inspect pagination and record findings

In [4]:
for link in soup.select('a[href]'):
    label = link.get_text(' ', strip=True).lower()
    classes = ' '.join(link.get('class', [])).lower()
    aria = (link.get('aria-label') or '').lower()
    if 'next' in label or 'next' in classes or 'next' in aria:
        print('Possible next link:', label, '| URL:', urljoin(response.url, link['href']))

print('Inspection timestamp (UTC):', datetime.now(timezone.utc).isoformat(timespec='seconds'))
print('IMPORTANT: verify the selector and next-page URL in your browser before running the full scraper.')

Possible next link: next page | URL: https://itch.io/games/top-sellers?page=2
Inspection timestamp (UTC): 2026-10-13T15:23:23+00:00
IMPORTANT: verify the selector and next-page URL in your browser before running the full scraper.


## 4. Browser Network tab
Open the Top Sellers page in a browser, press **F12 → Network**, reload, and inspect the main Document and Fetch/XHR requests. Record whether titles/prices are already in the HTML. Do not assume an API endpoint exists, and do not use authenticated/private endpoints. Fill in `reports/reconnaissance_notes.md` using your actual observations.

## 5. Next steps
Once the selectors are verified, test `src/scraper.py` with 100 items and a delay of at least 3 seconds. If zero cards are found, inspect the live HTML and update selectors first. Do not pretend that the scraper collected 500 records if it did not.